# Setup

In [1]:
%load_ext autoreload
%autoreload 2

# Pytorch import
import torch

#General import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import time
import re
from tqdm import tqdm
import phonetic_manipulation
import unicodedata
import itertools
import pickle
from praatio import textgrid
import scipy.special as special
import scipy.spatial as spatial
import scipy.stats as stats
from wordfreq import word_frequency
import os
import imageio
from sklearn.mixture import GaussianMixture
from matplotlib import cm
import matplotlib.gridspec as gridspec

# Bert import
from transformers import CamembertForMaskedLM, CamembertTokenizer, CamembertForCausalLM, AutoTokenizer, AutoConfig

# Custom import
import predict_utils as predict
from predict_utils import predict_mask, advance_mask, get_phn_token, phonemes_possibilities,log2normproba, logit2proba
from pingouin import bayesfactor_ttest

tokenizer = CamembertTokenizer.from_pretrained("camembert-base", truncation_side='left')
camembert = CamembertForMaskedLM.from_pretrained("camembert-base")
camembert.eval()
camembert_complete = CamembertForMaskedLM.from_pretrained("camembert-base", output_hidden_states = True, output_attentions  = True, return_dict = True)
camembert_complete.eval()
config = AutoConfig.from_pretrained("almanach/camembert-base")
config.is_decoder = True

bad_index_file = 'C:/Users/D-CAP/Documents/GitHub/witching-star/lexique_dataframe/index_foireux.pkl'
bad_index = pd.read_pickle(bad_index_file)
bad_index.remove(7885)
#bad_index.remove(9)


c:\Users\D-CAP\anaconda3\envs\Camembert\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Some weights of the model checkpoint at camembert-base were not used when initializing CamembertForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing CamembertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CamembertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSeque

# Generate Word Features from Text

### Compute Lexical Distributions via LLM

In [ ]:
from predict_utils import predict_mask, advance_mask ,log2normproba, logit2proba
sorciere2 = textgrid.openTextgrid('D:/Sorciere_v0-main/regressors/sorciere_reg_ling.textgrid', includeEmptyIntervals=False)
phones_grid = sorciere2.getTier('phones').entries
syllables_grid = sorciere2.getTier('syll').entries
words_grid = sorciere2.getTier('words').entries

with open('D:/Sorciere_v0-main/sorciere_wav/sorciere_1.txt','r+') as file:
    sorcieres = file.read()

proba_dict = dict()
pos = 0
context = '<mask>.'
end_context_list, onset_list, offset_list = [], [], []
for entry in words_grid:
    if entry[2] != '_':
        onset_list.append(entry[0])
        offset_list.append(entry[1])
        end_context_list.append(entry[2])


for next_word, onset, offset in zip(end_context_list, onset_list, offset_list):
    next_word_code = tokenizer.encode(next_word)[1:-1]
    encode_token_dict = dict()
    for token_index,next_token in enumerate(next_word_code):
        predict_token = tokenizer.decode(next_token)
        if (not next_token in bad_index) or (next_word == '_'):
            #print(context)
            output = predict_mask(context)
            output[bad_index] = -30
            output_proba = logit2proba(output)
            encode_token_dict[token_index] = {'token': predict_token, 'token_id': next_token, 'proba': output_proba[next_token], 'distribution': output_proba}
        context = advance_mask(context, predict_token, connector = ' ')
    
    if len(encode_token_dict) > 0:
        #proba_dict[re.sub('[^a-zA-Z]+', '', next_word) + str(pos)] = {'position': pos, 'token_distribution':encode_token_dict}
        proba_dict[next_word + '--' + str(pos)] = {'position': pos, 'token_distribution':encode_token_dict, 'onset': onset, 'offset': offset}
        pos += 1
    context = advance_mask(context, ' ', connector = '')
    if pos % 500 == 0:
        print(pos)
        print(context)


### Generate Features from Distributions

In [ ]:
with open(r'C:\Users\D-CAP\Documents\GitHub\witching-star\lexique_dataframe\word_regularity.pickle','rb') as file:
    wrd_regularity_dict = pickle.load(file)

In [ ]:
%matplotlib qt

sample_distrib_indices = [161,262,500]

log_base = np.exp(1)
renyi_values = np.asarray([0.01,0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1,1.1,1.2,1.3,1.4,1.5,2,5,10])
#renyi_values = np.hstack([np.logspace(-1.5,1.5,100)])
#renyi_values = np.hstack([np.logspace(-2,2,50)])
good_index = [not i in (bad_index + [7885]) for i in np.arange(32005)]
#renyi_values = np.asarray([1,2])

word_list = []
renyi_list = []
min_entropy_list = []
max_entropy_list = []
min2_entropy_list = []
min12_entropy_list = []
min15_entropy_list = []
max2_entropy_list = []
collision_entropy_list = []
shannon_entropy_list = []
hart_entropy_list = []
sample_distrib_list = []
surprisal_list = []
token_list = []
token_id_list = []
fluctuation_list = []
lexical_surprise_list = []
word_item_list = []
most_likely_token_list = []
most_likely_token2_list = []
most_likely_token3_list = []
most_likely_token4_list = []
most_likely_token5_list = []


onset_list = []
offset_list = []
for word in proba_dict:
    word_text = word.split('--')[0]
    if word_text != '_':
        onset, offset = proba_dict[word]['onset'], proba_dict[word]['offset']
        encode_tokens_dict = proba_dict[word]['token_distribution']
        proba_token = []
        surprisal = 0
        if word_text in wrd_regularity_dict:
            word_freq = wrd_regularity_dict[word_text]
        else:
            word_freq = word_frequency(word_text, 'fr')
            #print(word_text, word_freq)
        #word_freq = word_frequency(word_text, 'fr')
        lexical_surprise = -np.log(word_freq)
        for token_index in encode_tokens_dict:
            token_info = encode_tokens_dict[token_index]
            probability_token = token_info['proba'] 
            surprisal += -adjusted_log(probability_token, log_base=log_base)/len(encode_tokens_dict)

            all_proba = token_info['distribution'] 
            all_token_indices = np.arange(len(all_proba))
            all_proba = all_proba[good_index]
            all_token_indices =  all_token_indices[good_index]
            all_proba = all_proba/np.sum(all_proba)
            all_proba2 = np.sort(all_proba)[:-1] / np.sum(np.sort(all_proba)[:-1])
            all_proba3 = np.sort(all_proba)[:-2] / np.sum(np.sort(all_proba)[:-2])
            if token_index == 0:
                first_token = token_info['token']
                first_token_id = token_info['token_id']
                most_likely_token_index = np.argmax(all_proba)
                most_likely_token = all_token_indices[most_likely_token_index]
                most_likely_token_index2 = np.argsort(all_proba)[-2]
                most_likely_token2 = all_token_indices[most_likely_token_index2]       
                most_likely_token_index3 = np.argsort(all_proba)[-3]
                most_likely_token3 = all_token_indices[most_likely_token_index3]           
                most_likely_token_index4 = np.argsort(all_proba)[-4]
                most_likely_token4 = all_token_indices[most_likely_token_index4]      
                most_likely_token_index5 = np.argsort(all_proba)[-5]
                most_likely_token5 = all_token_indices[most_likely_token_index5]    
                all_surprisal = -adjusted_log(all_proba, log_base=log_base)
                all_surprisal2 = -adjusted_log(all_proba2, log_base=log_base)
                all_surprisal3 = -adjusted_log(all_proba3, log_base=log_base)
                fluctuation = np.sqrt(np.sum(all_proba * all_surprisal**2) - np.sum(all_proba * all_surprisal)**2)
                sample_distrib = all_proba
                renyi = []
                for renyi_value in renyi_values:
                    renyi.append(renyi_entropy(all_proba,renyi_value, log_base=log_base))
                min_entropy = np.sort(all_surprisal)[0]
                min2_entropy = np.sort(all_surprisal)[1]
                min12_entropy = -np.log(np.sum(np.sort(all_proba)[::-1][:2]))
                min15_entropy = -np.log(np.sum(np.sort(all_proba)[::-1][:5]))
                max_entropy = renyi_entropy(all_proba2,1, log_base=log_base)
                max2_entropy = renyi_entropy(all_proba3,1, log_base=log_base)
                collision_entropy = renyi_entropy(all_proba,2, log_base=log_base)
                shannon_entropy = renyi_entropy(all_proba,1, log_base=log_base)
                hart_entropy = renyi_entropy(all_proba,0, log_base=log_base)
            
            if len(all_proba[all_proba==0]) > 0:
                print('problem')

        most_likely_token_list.append(most_likely_token)
        most_likely_token2_list.append(most_likely_token2)
        most_likely_token3_list.append(most_likely_token3)
        most_likely_token4_list.append(most_likely_token4)
        most_likely_token5_list.append(most_likely_token5)
        token_list.append(first_token)
        token_id_list.append(first_token_id)
        word_list.append(word_text)
        word_item_list.append(word)
        renyi_list.append(renyi)
        min_entropy_list.append(min_entropy)
        min2_entropy_list.append(min2_entropy)
        min12_entropy_list.append(min12_entropy)
        min15_entropy_list.append(min15_entropy)
        max_entropy_list.append(max_entropy)
        max2_entropy_list.append(max2_entropy)
        collision_entropy_list.append(collision_entropy)
        shannon_entropy_list.append(shannon_entropy)
        hart_entropy_list.append(hart_entropy)
        surprisal_list.append(surprisal)
        fluctuation_list.append(fluctuation)
        lexical_surprise_list.append(lexical_surprise)
        
        onset_list.append(onset)
        offset_list.append(offset)

        if len(onset_list) in sample_distrib_indices:
            sample_distrib_list.append(sample_distrib)
hart_entropy_list = renyi_list[0]
#renyi_list = renyi_list + min2_entropy_list + min2_entropy_list + max_entropy_list + max2_entropy_list + shannon_entropy_list + surprisal_list + obvious_fluctuation_list
renyi_array = np.asarray(renyi_list)
n_renyi = renyi_array.shape[1]
renyi_array = np.hstack([renyi_array,np.asarray(min_entropy_list)[:,np.newaxis] ,np.asarray(min2_entropy_list)[:,np.newaxis] ,
                         np.asarray(min12_entropy_list)[:,np.newaxis] , np.asarray(min15_entropy_list)[:,np.newaxis] ,
                         np.asarray(shannon_entropy_list)[:,np.newaxis] ,np.asarray(surprisal_list)[:,np.newaxis] , np.asarray(lexical_surprise_list)[:,np.newaxis]])


### Save Features

In [ ]:
# Semantic Data
path_semantic = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/predictive_regressors_wd.pkl"
semantic_data = pickle.load(open(path_semantic, 'rb'))
wrd_fs = semantic_data['fs']
duration = semantic_data['X'].shape[0]/semantic_data['fs']

In [ ]:
# Fill dictionary
fs = 1000
semantic_regressor = dict()
semantic_regressor['fs'] = fs
semantic_regressor['names'] = ['renyi ' + str(i) for i in renyi_values] + ['MinEntro','Min2Entro','Min15Entro','Max2Entro', 'Shannon Entropy', 'Surprisal', 'Lexical Surprisal']
semantic_regressor['timing_info'] = np.vstack([np.asarray(onset_list), np.asarray(offset_list)]).T
semantic_regressor['content_info'] = word_list

X = np.zeros([int(duration * fs),renyi_array.shape[1]])
for renyi_index in range(renyi_array.shape[1]):
    for onset, renyi in zip(onset_list, renyi_array[:,renyi_index]):
        X[int(onset * fs),renyi_index] = renyi


semantic_regressor['X'] = X

In [ ]:
filename = 'semantic_regressor/renyi_wrdonly_array.pickle'
with open(filename, 'wb') as file:
    pickle.dump(semantic_regressor, file)

### Compute Quartiles

In [ ]:
duration_min, duration_max = 0.1, 0.7

duration_list = np.asarray(offset_list) - np.asarray(onset_list)
duration_mask = list((duration_list > duration_min) * (duration_list < duration_max))

onset_array = np.asarray(onset_list)[duration_mask]
offset_array = np.asarray(offset_list)[duration_mask]

In [ ]:
token_choice = 72 #72

confidence_list = np.sort(np.arange(len(renyi_array[duration_mask,20]))[np.argsort(renyi_array[duration_mask,20])[:token_choice ]])
inconfidence_list = np.sort(np.arange(len(renyi_array[duration_mask,20]))[np.argsort(renyi_array[duration_mask,20])[-token_choice :]])

uncertainty_list = np.sort(np.arange(len(renyi_array[duration_mask,0]))[np.argsort(renyi_array[duration_mask,0])[:token_choice ]])
certainty_list = np.sort(np.arange(len(renyi_array[duration_mask,0]))[np.argsort(renyi_array[duration_mask,0])[-token_choice :]])

confidence_onsets = onset_array[confidence_list]
confidence_offsets = offset_array[confidence_list]
inconfidence_onsets = onset_array[inconfidence_list]
inconfidence_offsets = offset_array[inconfidence_list]

certainty_onsets = onset_array[certainty_list]
certainty_offsets = offset_array[certainty_list]
uncertainty_onsets = onset_array[uncertainty_list]
uncertainty_offsets = offset_array[uncertainty_list]

In [ ]:
surprisal_array = np.asarray(surprisal_list)[duration_mask]
confident_surprisal_list = surprisal_array[confidence_list]
inconfident_surprisal_list = surprisal_array[inconfidence_list]
uncertain_surprisal_list = surprisal_array[uncertainty_list]
certain_surprisal_list = surprisal_array[certainty_list]

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, scale

# Assuming data is in a 2D array where column 0 = A, column 1 = B
data = np.vstack([renyi_array[duration_mask,0],surprisal_array]).T
data = np.vstack([renyi_array[duration_mask,20],surprisal_array]).T
#data = np.vstack([renyi_array[duration_mask,0],renyi_array[duration_mask,20]]).T
#data = np.vstack([np.asarray(max_entropy_list)[duration_mask],surprisal_array]).T
#data = np.vstack([renyi_array[duration_mask,11],surprisal_array]).T

# Standardize the features
scaler = StandardScaler()
data_std = scaler.fit_transform(data)

# Perform PCA
pca = PCA(n_components=2)
data_pca = pca.fit_transform(data_std)
data_pca = data_std

color_quartiles = ['turquoise', 'darkblue', 'mediumaquamarine', 'teal']
color_quartiles = ['gold', 'darkred', 'orange', 'salmon']

label = 'Dispersion'
label = 'Strength'

hard_lim = [10.25,10.29,2.7,18] #[10.25,10.29,1,12]
hard_lim = [0.2,3.5,2.7,18]
#hard_lim = [10.25,10.29,0.6,2.5]
#hard_lim = [2.5,6.2,2.7,18]

# Define thresholds (e.g., medians)
quant1, quant2 = 45, 45
quant1, quant2 = 45, 45
#quant1, quant2 = 45, 45
#quant1, quant2 = 45, 45
#quant1, quant2 = 45, 45

quant3, quant4 = 70, 70
quant3, quant4 = 70, 70
#quant3, quant4 = 55, 55
#quant3, quant4 = 70, 70
#quant3, quant4 = 70, 70

choice_reg = 9042
choice_reg = 8927

pc1_min = np.percentile(data_pca[:, 0],quant1)
pc2_min = np.percentile(data_pca[:, 1],quant2)
pc1_max = np.percentile(data_pca[:, 0],quant3)
pc2_max = np.percentile(data_pca[:, 1],quant4)

# Assign samples to quadrants
Q1 = (data_pca[:, 0] <= pc1_min) & (data_pca[:, 1] <= pc2_min)  # Low-PC1 + Low-PC2
Q2 = (data_pca[:, 0] > pc1_max) & (data_pca[:, 1] > pc2_max)    # High-PC1 + High-PC2
Q3 = (data_pca[:, 0] > pc1_max) & (data_pca[:, 1] <= pc2_min)   # High-PC1 + Low-PC2
Q4 = (data_pca[:, 0] <= pc1_min) & (data_pca[:, 1] > pc2_max)   # Low-PC1 + High-PC2

Q1[data[:,0] < hard_lim[0]],Q2[data[:,0] < hard_lim[0]],Q3[data[:,0] < hard_lim[0]],Q4[data[:,0] < hard_lim[0]] = False,False,False,False
Q1[data[:,0] > hard_lim[1]], Q2[data[:,0] > hard_lim[1]], Q3[data[:,0] > hard_lim[1]], Q4[data[:,0] > hard_lim[1]] = False,False,False,False
Q1[data[:,1] < hard_lim[2]],Q2[data[:,1] < hard_lim[2]],Q3[data[:,1] < hard_lim[2]],Q4[data[:,1] < hard_lim[2]] = False,False,False,False
Q1[data[:,1] > hard_lim[3]],Q2[data[:,1] > hard_lim[3]],Q3[data[:,1] > hard_lim[3]],Q4[data[:,1] > hard_lim[3]] = False,False,False,False


# Get the indices for each quadrant
'''
indices_Q1 = np.where(Q1)[0][np.argsort(np.linalg.norm(data_pca[Q1],axis = 1, ord = 0))][-token_choice:]
indices_Q2 = np.where(Q2)[0][np.argsort(np.linalg.norm(data_pca[Q2],axis = 1, ord = 0))][-token_choice:]
indices_Q3 = np.where(Q3)[0][np.argsort(np.linalg.norm(data_pca[Q3],axis = 1, ord = 0))][-token_choice:]
indices_Q4 = np.where(Q4)[0][np.argsort(np.linalg.norm(data_pca[Q4],axis = 1, ord = 0))][-token_choice:]


indices_Q1 = np.where(Q1)[0][np.argsort(np.max(np.abs(data_pca[Q1]),axis = 1))][-token_choice:]
indices_Q2 = np.where(Q2)[0][np.argsort(np.max(np.abs(data_pca[Q2]),axis = 1))][-token_choice:]
indices_Q3 = np.where(Q3)[0][np.argsort(np.max(np.abs(data_pca[Q3]),axis = 1))][-token_choice:]
indices_Q4 = np.where(Q4)[0][np.argsort(np.max(np.abs(data_pca[Q4]),axis = 1))][-token_choice:]

'''

indices_Q1 = np.where(Q1)[0][-token_choice:]
indices_Q2 = np.where(Q2)[0][-token_choice:]
indices_Q3 = np.where(Q3)[0][-token_choice:]
indices_Q4 = np.where(Q4)[0][-token_choice:]

objective = 0.30



'''
for i in [8927] + list(np.arange(10000,20000)):
    np.random.seed(i)
    indices_Q1 = np.random.permutation(np.where(Q1)[0])[-token_choice:]
    indices_Q2 = np.random.permutation(np.where(Q2)[0])[-token_choice:]
    indices_Q3 = np.random.permutation(np.where(Q3)[0])[-token_choice:]
    indices_Q4 = np.random.permutation(np.where(Q4)[0])[-token_choice:]

    bayes_0_1_4 = bayesfactor_ttest(stats.ttest_ind(data[indices_Q1,0], data[indices_Q4,0])[0], len(indices_Q1), len(indices_Q4))
    bayes_0_2_3 = bayesfactor_ttest(stats.ttest_ind(data[indices_Q2,0], data[indices_Q3,0])[0], len(indices_Q2), len(indices_Q3))
    bayes_1_1_3 = bayesfactor_ttest(stats.ttest_ind(data[indices_Q1,1], data[indices_Q3,1])[0], len(indices_Q1), len(indices_Q3))
    bayes_1_2_4 = bayesfactor_ttest(stats.ttest_ind(data[indices_Q1,1], data[indices_Q3,1])[0], len(indices_Q1), len(indices_Q3))

    bayes_objective = np.max([bayes_0_1_4, bayes_1_1_3, bayes_0_2_3, bayes_1_2_4])
    if bayes_objective < objective:
        choice_reg = i
        objective = bayes_objective
        print(bayes_0_1_4, bayes_1_1_3, bayes_0_2_3, bayes_1_2_4)
print(choice_reg, objective)
'''

np.random.seed(choice_reg)
indices_Q1 = np.random.permutation(np.where(Q1)[0])[-token_choice:]
indices_Q2 = np.random.permutation(np.where(Q2)[0])[-token_choice:]
indices_Q3 = np.random.permutation(np.where(Q3)[0])[-token_choice:]
indices_Q4 = np.random.permutation(np.where(Q4)[0])[-token_choice:]

bayes_0_1_4 = bayesfactor_ttest(stats.ttest_ind(data[indices_Q1,0], data[indices_Q4,0])[0], len(indices_Q1), len(indices_Q4))
bayes_0_2_3 = bayesfactor_ttest(stats.ttest_ind(data[indices_Q2,0], data[indices_Q3,0])[0], len(indices_Q2), len(indices_Q3))
bayes_1_1_3 = bayesfactor_ttest(stats.ttest_ind(data[indices_Q1,1], data[indices_Q3,1])[0], len(indices_Q1), len(indices_Q3))
bayes_1_2_4 = bayesfactor_ttest(stats.ttest_ind(data[indices_Q1,1], data[indices_Q3,1])[0], len(indices_Q1), len(indices_Q3))


print(len(indices_Q1), len(indices_Q2), len(indices_Q3), len(indices_Q4))




fig, ax = plt.subplots(figsize = (7,6))
ax.scatter(scale(data[:,0]), scale(data[:,1]), color = 'k', alpha = 0.15, s = 50)
ax.scatter(scale(data[:,0])[indices_Q1], scale(data[:,1])[indices_Q1], edgecolors='k', s = 125, color = color_quartiles[0])
ax.scatter(scale(data[:,0])[indices_Q2], scale(data[:,1])[indices_Q2], edgecolors='k', s = 125, color = color_quartiles[1])
ax.scatter(scale(data[:,0])[indices_Q3], scale(data[:,1])[indices_Q3], edgecolors='k', s = 125, color = color_quartiles[2])
ax.scatter(scale(data[:,0])[indices_Q4], scale(data[:,1])[indices_Q4], edgecolors='k', s = 125, color = color_quartiles[3])

ax.spines[['right', 'top']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(2)
ax.tick_params(width=3, labelsize = 16)
ax.set_ylabel('Surprisal (z-score)', size = 20)
ax.set_xlabel(label + ' (z-score)', size = 20)
ax.set_xlim(-2.5,2.5)
ax.set_ylim(-1.82,3.92)
fig.tight_layout()


In [ ]:
Q1_onsets = onset_array[indices_Q1]
Q2_onsets = onset_array[indices_Q2]
Q3_onsets = onset_array[indices_Q3]
Q4_onsets = onset_array[indices_Q4]

Q1_offsets = offset_array[indices_Q1]
Q2_offsets = offset_array[indices_Q2]
Q3_offsets = offset_array[indices_Q3]
Q4_offsets = offset_array[indices_Q4]

Q1_durations = Q1_offsets - Q1_onsets
Q2_durations = Q2_offsets - Q2_onsets
Q3_durations = Q3_offsets - Q3_onsets
Q4_durations = Q4_offsets - Q4_onsets

Q1_surprisal_list = surprisal_array[indices_Q1]
Q2_surprisal_list = surprisal_array[indices_Q2]
Q3_surprisal_list = surprisal_array[indices_Q3]
Q4_surprisal_list = surprisal_array[indices_Q4]

In [ ]:
# Semantic Data
path_semantic = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/predictive_regressors_wd.pkl"
semantic_data = pickle.load(open(path_semantic, 'rb'))
wrd_fs = semantic_data['fs']
duration = semantic_data['X'].shape[0]/semantic_data['fs']

In [ ]:
# Fill dictionary
fs = 1000
semantic_regressor = dict()
semantic_regressor['fs'] = fs
semantic_regressor['names'] = ['Q1']
semantic_regressor['timing_info'] = np.vstack([np.asarray(Q1_onsets), np.asarray(Q1_offsets)]).T
semantic_regressor['content_info'] = word_list

X = np.zeros([int(duration * fs),2])
for Q1_onset, Q1_offset, Q1_surprisal in zip(Q1_onsets, Q1_offsets, Q1_surprisal_list):
    X[int(Q1_onset * fs),0] = Q1_surprisal
    X[int(Q1_offset * fs),1] = Q1_surprisal

semantic_regressor['X'] = X

In [ ]:
filename = 'semantic_regressor/Q1_reg.pickle'
with open(filename, 'wb') as file:
    pickle.dump(semantic_regressor, file)